In [1]:
import open3d as o3d

dataset_name = "stereo_budha_charuco"
DATASETS_PATH = "datasets"

pcd_path = f"{DATASETS_PATH}/{dataset_name}/data/results/buddha-pcd-cleaned.ply"

pcd = o3d.io.read_point_cloud(pcd_path)
print(pcd)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
PointCloud with 3136190 points.


In [2]:
down_pcd = pcd.voxel_down_sample(voxel_size=2)
print(down_pcd)

PointCloud with 114539 points.


In [3]:
import numpy as np

down_pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=10.0, max_nn=30)
)

In [4]:
down_pcd.orient_normals_consistent_tangent_plane(100)

In [5]:
depth = 9
with o3d.utility.VerbosityContextManager(o3d.utility.VerbosityLevel.Debug) as cm:
    buddha_mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        down_pcd, depth=depth, linear_fit=True
    )

[Open3D DEBUG] Input Points / Samples: 114539 / 114504
[Open3D DEBUG] #   Got kernel density: 0.11496806144714355 (s), 574.44921875 (MB) / 574.44921875 (MB) / 890 (MB)
[Open3D DEBUG] #     Got normal field: 0.3424820899963379 (s), 611.375 (MB) / 611.375 (MB) / 890 (MB)
[Open3D DEBUG] Point weight / Estimated Area: 3.730639e-05 / 4.273037e+00
[Open3D DEBUG] #       Finalized tree: 0.3346679210662842 (s), 616.5 (MB) / 616.5 (MB) / 890 (MB)
[Open3D DEBUG] #  Set FEM constraints: 0.4156379699707031 (s), 616.5 (MB) / 616.5 (MB) / 890 (MB)
[Open3D DEBUG] #Set point constraints: 0.12106704711914062 (s), 616.5 (MB) / 616.5 (MB) / 890 (MB)
[Open3D DEBUG] Leaf Nodes / Active Nodes / Ghost Nodes: 1927927 / 1298912 / 904433
[Open3D DEBUG] Memory Usage: 616.500 MB
Cycle[0] Depth[0/9]:	Updated constraints / Got system / Solved in:  0.000 /  0.000 /  0.000	(616.629 MB)	Nodes: 8
CG: 1.7212e+00 -> 1.7212e+00 -> 8.5271e-04 (5.0e-04) [0]
[Open3D DEBUG] # Linear system solved: 1.3416168689727783 (s), 616.

In [ ]:
vertices_to_remove = densities < np.quantile(densities, 0.01)
buddha_mesh.remove_vertices_by_mask(vertices_to_remove)
buddha_mesh.paint_uniform_color([0.8, 0.8, 0.8])

mesh_path = f"{DATASETS_PATH}/{dataset_name}/data/results/mesh.ply"

o3d.io.write_triangle_mesh(mesh_path, buddha_mesh)

True